# Split FS-TACRED Train Into Feedback and Heldout Fitness

This notebook splits `data_few_shot/_train_data.json` by relation names, not by individual examples.

Output files:

- `data_few_shot/_train_feedback_data.json`
- `data_few_shot/_train_heldout_data.json`
- `data_few_shot/train_feedback_heldout_relation_split.json`

Default split:

- 19 positive train relations for feedback
- 6 positive train relations for heldout fitness
- `no_relation` is kept as a special NOTA/background key, not counted in 19/6

In [18]:
import json
import random
from collections import Counter
from pathlib import Path

In [ ]:
# Configure paths and split size.
# This notebook lives in ../codes, so the FS-TACRED transformation folder is ./Few_Shot_transformation_and_sampling.
BASE_DIR = Path("Few_Shot_transformation_and_sampling")
DATA_DIR = BASE_DIR / "data_few_shot"

TRAIN_DATA_PATH = DATA_DIR / "_train_data.json"
DEV_DATA_PATH = DATA_DIR / "_dev_data.json"
FEEDBACK_OUTPUT_PATH = DATA_DIR / "_train_feedback_data.json"
HELDOUT_OUTPUT_PATH = DATA_DIR / "_train_heldout_data.json"
SPLIT_OUTPUT_PATH = DATA_DIR / "train_feedback_heldout_relation_split.json"

SEED = 170000 # ran SEED+333 and SEED+100 in some codes below to solve the issue with a relation that gets only 1 instance split
N_FEEDBACK_RELATIONS = 19
N_HELDOUT_RELATIONS = 6
NO_RELATION = "no_relation"

In [20]:
with open(DEV_DATA_PATH , "r") as f:
    dev_data = json.load(f)

In [21]:
rr = set()
for r in dev_data["no_relation"]:
    rr.add(r["relation"])
print(len(rr))

36


In [22]:
with open(TRAIN_DATA_PATH, "r") as f:
    train_data = json.load(f)

positive_relations = [rel for rel in train_data.keys() if rel != NO_RELATION]

print("train top-level keys:", len(train_data))
print("positive train relations:", len(positive_relations))
print("no_relation examples:", len(train_data[NO_RELATION]))

assert NO_RELATION in train_data
assert len(positive_relations) == N_FEEDBACK_RELATIONS + N_HELDOUT_RELATIONS

train top-level keys: 26
positive train relations: 25
no_relation examples: 59961


In [40]:
# Deterministically split relation names.
# Change SEED above if you want a different random relation split.
rng = random.Random(SEED+100)
shuffled_relations = positive_relations.copy()
rng.shuffle(shuffled_relations)

heldout_relations = sorted(shuffled_relations[:N_HELDOUT_RELATIONS])
feedback_relations = sorted(shuffled_relations[N_HELDOUT_RELATIONS:])

print("feedback relations:", len(feedback_relations))
for rel in feedback_relations:
    print("  ", rel, len(train_data[rel]))

print("\nheldout relations:", len(heldout_relations))
for rel in heldout_relations:
    print("  ", rel, len(train_data[rel]))

assert set(feedback_relations).isdisjoint(heldout_relations)
assert set(feedback_relations) | set(heldout_relations) == set(positive_relations)

feedback relations: 19
   org:alternate_names 808
   org:city_of_headquarters 382
   org:members 170
   org:number_of_employees/members 75
   org:political/religious_affiliation 105
   org:stateorprovince_of_headquarters 229
   org:website 111
   per:cause_of_death 117
   per:charges 72
   per:city_of_birth 65
   per:countries_of_residence 445
   per:country_of_birth 28
   per:country_of_death 6
   per:date_of_death 134
   per:employee_of 1524
   per:other_family 179
   per:spouse 258
   per:stateorprovince_of_birth 38
   per:title 2443

heldout relations: 6
   org:dissolved 23
   org:shareholders 76
   org:subsidiaries 296
   per:cities_of_residence 374
   per:parents 152
   per:religion 53


## Build Feedback and Heldout Data

Every top-level relation key is split deterministically at the example level using the 19/6 feedback/heldout ratio.

For feedback relations, the feedback slice stays under its positive relation key and the heldout slice goes into heldout `no_relation`. For heldout relations, the heldout slice stays positive and the feedback slice goes into feedback `no_relation`.

In [41]:
def split_examples_for_feedback_heldout(examples, seed_suffix):
    examples = list(examples)
    split_rng = random.Random(f"{SEED+333}:{seed_suffix}")
    split_rng.shuffle(examples)
    n_feedback = round(
        len(examples) * N_FEEDBACK_RELATIONS / (N_FEEDBACK_RELATIONS + N_HELDOUT_RELATIONS)
    )
    return examples[:n_feedback], examples[n_feedback:]


feedback_data = {NO_RELATION: []}
heldout_data = {NO_RELATION: []}

for rel, examples in train_data.items():
    feedback_examples, heldout_examples = split_examples_for_feedback_heldout(examples, rel)

    if rel == NO_RELATION:
        feedback_data[NO_RELATION].extend(feedback_examples)
        heldout_data[NO_RELATION].extend(heldout_examples)
    elif rel in feedback_relations:
        feedback_data[rel] = feedback_examples
        heldout_data[NO_RELATION].extend(heldout_examples)
    elif rel in heldout_relations:
        feedback_data[NO_RELATION].extend(feedback_examples)
        heldout_data[rel] = heldout_examples
    else:
        raise ValueError(f"Unexpected relation key: {rel}")

print("feedback top-level keys:", len(feedback_data))
print("feedback examples:", sum(len(v) for v in feedback_data.values()))
print("feedback no_relation examples:", len(feedback_data[NO_RELATION]))

print("\nheldout top-level keys:", len(heldout_data))
print("heldout examples:", sum(len(v) for v in heldout_data.values()))
print("heldout no_relation examples:", len(heldout_data[NO_RELATION]))

feedback top-level keys: 20
feedback examples: 51773
feedback no_relation examples: 46310

heldout top-level keys: 7
heldout examples: 16351
heldout no_relation examples: 16117


In [42]:
# Sanity checks for schema consistency and disjoint example assignment.
assert set(feedback_data) == {NO_RELATION, *feedback_relations}
assert set(heldout_data) == {NO_RELATION, *heldout_relations}

feedback_positive_actual_relations = {
    ex["relation"]
    for rel, examples in feedback_data.items()
    if rel != NO_RELATION
    for ex in examples
}
heldout_positive_actual_relations = {
    ex["relation"]
    for rel, examples in heldout_data.items()
    if rel != NO_RELATION
    for ex in examples
}

assert feedback_positive_actual_relations == set(feedback_relations)
assert heldout_positive_actual_relations == set(heldout_relations)

source_ids = [ex["id"] for examples in train_data.values() for ex in examples]
feedback_ids = [ex["id"] for examples in feedback_data.values() for ex in examples]
heldout_ids = [ex["id"] for examples in heldout_data.values() for ex in examples]

assert len(source_ids) == len(set(source_ids)), "source train data has duplicate ids"
assert set(feedback_ids).isdisjoint(heldout_ids), "feedback and heldout examples overlap"
assert set(feedback_ids) | set(heldout_ids) == set(source_ids), "split lost or added examples"

feedback_no_relation_actual_counts = Counter(ex["relation"] for ex in feedback_data[NO_RELATION])
heldout_no_relation_actual_counts = Counter(ex["relation"] for ex in heldout_data[NO_RELATION])

print("feedback positive actual relation labels:", len(feedback_positive_actual_relations))
print("heldout positive actual relation labels:", len(heldout_positive_actual_relations))
print("feedback no_relation internal labels:", len(feedback_no_relation_actual_counts))
print("heldout no_relation internal labels:", len(heldout_no_relation_actual_counts))
print("feedback/heldout id overlap:", len(set(feedback_ids) & set(heldout_ids)))

feedback positive actual relation labels: 19
heldout positive actual relation labels: 6
feedback no_relation internal labels: 23
heldout no_relation internal labels: 36
feedback/heldout id overlap: 0


In [43]:
split_metadata = {
    "seed": SEED,
    "no_relation_key": NO_RELATION,
    "source_train_data": str(TRAIN_DATA_PATH),
    "feedback_output": str(FEEDBACK_OUTPUT_PATH),
    "heldout_output": str(HELDOUT_OUTPUT_PATH),
    "n_feedback_relations": len(feedback_relations),
    "n_heldout_relations": len(heldout_relations),
    "feedback_relations": feedback_relations,
    "heldout_relations": heldout_relations,
}

with open(FEEDBACK_OUTPUT_PATH, "w") as f:
    json.dump(feedback_data, f)

with open(HELDOUT_OUTPUT_PATH, "w") as f:
    json.dump(heldout_data, f)

with open(SPLIT_OUTPUT_PATH, "w") as f:
    json.dump(split_metadata, f, indent=2)

print("wrote:", FEEDBACK_OUTPUT_PATH)
print("wrote:", HELDOUT_OUTPUT_PATH)
print("wrote:", SPLIT_OUTPUT_PATH)

wrote: Few_Shot_transformation_and_sampling/data_few_shot/_train_feedback_data.json
wrote: Few_Shot_transformation_and_sampling/data_few_shot/_train_heldout_data.json
wrote: Few_Shot_transformation_and_sampling/data_few_shot/train_feedback_heldout_relation_split.json


## Optional: Generate Heldout Fitness Episodes

Run this after creating `_train_heldout_data.json` if you want 3,000 one-query heldout fitness cases.

In [ ]:
# !python Few_Shot_transformation_and_sampling/episodes_sampling_for_fs_TACRED.py \
#   --file_name Few_Shot_transformation_and_sampling/data_few_shot/_train_heldout_data.json \
#   --episodes_size 1000 \
#   --N 5 \
#   --K 1 \
#   --number_of_queries 3 \
#   --seed 160300 \
#   --output_file_name Few_Shot_transformation_and_sampling/fs_tacred_train_heldout_5_way_1_shots_1000episodes_160300.json

In [1]:
!python Few_Shot_transformation_and_sampling/episodes_sampling_for_fs_TACRED.py \
  --file_name Few_Shot_transformation_and_sampling/data_few_shot/_train_heldout_data.json \
  --episodes_size 6000 \
  --N 5 \
  --K 1 \
  --number_of_queries 1 \
  --seed 170000 \
  --output_file_name Few_Shot_transformation_and_sampling/fs_tacred_train_heldout_5_way_1_shots_6000episodes_q1_170000.json